<a href="https://colab.research.google.com/github/matidmax-dot/AutoFake/blob/main/App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Instalar librerías
!pip install -q fastapi uvicorn pandas python-multipart

# 2. Descargar Cloudflare Tunnel
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

import os
import re
import sqlite3
import threading
import subprocess
import time
from google.colab import drive
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import uvicorn

print("📌 Paso 1: Conectando a Google Drive...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

DB_PATH = '/content/vehiculos.db'

print("\n📌 Paso 2: Verificando base de datos...")
if not os.path.exists(DB_PATH):
    archivo_tar = None
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'vehiculos.db.tar.gz' in files:
            archivo_tar = os.path.join(root, 'vehiculos.db.tar.gz')
            break

    if archivo_tar:
        print(f"   Descomprimiendo en segundo plano...")
        !tar -xf "{archivo_tar}" -C /content/
        print("   ¡Descompresión completada!")

if not os.path.exists(DB_PATH):
    for root, dirs, files in os.walk('/content'):
        if 'vehiculos.db' in files:
            DB_PATH = os.path.join(root, 'vehiculos.db')
            break

print("\n📌 Paso 3: Lanzando Servidor de la App con Logos...")
app = FastAPI()

HTML_CODE = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Alerta Vehículo PRT</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style> body { background-color: #0f172a; color: #f8fafc; font-family: system-ui, sans-serif; } </style>
</head>
<body class="p-4 max-w-lg mx-auto">

    <!-- Encabezado -->
    <div class="bg-slate-800 p-4 rounded-2xl shadow-lg border border-slate-700 text-center mb-4">
        <h1 class="text-xl font-bold text-amber-400">Alerta Vehículo PRT</h1>
        <p class="text-xs text-slate-400">Historial Técnico & Detección de Fraude</p>
    </div>

    <!-- Buscador -->
    <div class="bg-slate-800 p-4 rounded-2xl shadow-lg border border-slate-700 mb-4">
        <div class="flex gap-2">
            <input type="text" id="patenteInput" placeholder="Ingresa Patente (ej: PYCK65)" class="w-full bg-slate-900 border border-slate-700 text-white font-bold text-center text-lg p-3 rounded-xl uppercase tracking-widest focus:outline-none focus:border-amber-400">
            <button onclick="buscarPatente()" class="bg-amber-500 text-slate-950 font-bold px-5 py-3 rounded-xl hover:bg-amber-400 transition">BUSCAR</button>
        </div>
    </div>

    <!-- Resultados -->
    <div id="resultados" class="hidden space-y-4">

        <!-- Alerta Fraude -->
        <div id="alertaFraude" class="hidden bg-red-950/80 border border-red-600 p-4 rounded-2xl text-red-200">
            <h2 class="font-bold text-red-400 flex items-center gap-2 text-sm uppercase tracking-wide">
                ⚠️ ALERTA DE ODÓMETRO / RIESGO DETECTADO
            </h2>
            <p id="alertaTexto" class="text-xs mt-2 leading-relaxed"></p>
        </div>

        <!-- Ficha Técnica con LOGO -->
        <div class="bg-slate-800 p-4 rounded-2xl border border-slate-700 relative overflow-hidden">
            <div class="flex justify-between items-start mb-3">
                <div>
                    <h3 class="text-xs font-bold text-slate-400 uppercase tracking-wider mb-1">Ficha Técnica del Vehículo</h3>
                    <h2 id="vehiculoTitulo" class="text-xl font-bold text-white"></h2>
                </div>
                <!-- LOGO DE LA MARCA -->
                <div id="logoContainer" class="bg-slate-900/80 p-2 rounded-xl border border-slate-700/80 flex items-center justify-center min-w-[52px] min-h-[52px]">
                    <img id="marcaLogo" src="" alt="Logo Marca" class="w-10 h-10 object-contain" onerror="document.getElementById('logoContainer').style.display='none'">
                </div>
            </div>

            <div class="grid grid-cols-2 gap-3 text-xs">
                <div class="bg-slate-900/60 p-2.5 rounded-xl">
                    <span class="text-slate-400 block">N° Motor:</span>
                    <strong id="numMotor" class="text-slate-200"></strong>
                </div>
                <div class="bg-slate-900/60 p-2.5 rounded-xl">
                    <span class="text-slate-400 block">N° Chasis / VIN:</span>
                    <strong id="numChasis" class="text-slate-200"></strong>
                </div>
            </div>

            <div class="grid grid-cols-3 gap-2 mt-3 text-center">
                <div class="bg-slate-900/60 p-2 rounded-xl">
                    <span id="cantAprobadas" class="block font-bold text-emerald-400 text-lg">0</span>
                    <span class="text-[10px] text-slate-400">Aprobadas</span>
                </div>
                <div class="bg-slate-900/60 p-2 rounded-xl">
                    <span id="cantRechazadas" class="block font-bold text-red-400 text-lg">0</span>
                    <span class="text-[10px] text-slate-400">Rechazadas</span>
                </div>
                <div class="bg-slate-900/60 p-2 rounded-xl">
                    <span id="tasaAprobacion" class="block font-bold text-cyan-400 text-lg">0%</span>
                    <span class="text-[10px] text-slate-400">Tasa Aprobación</span>
                </div>
            </div>
        </div>

        <!-- Gráfico -->
        <div class="bg-slate-800 p-4 rounded-2xl border border-slate-700">
            <h3 class="text-xs font-bold text-slate-400 uppercase tracking-wider mb-3">Evolución del Kilometraje</h3>
            <canvas id="kmChart" class="w-full max-h-48"></canvas>
        </div>

        <!-- Historial -->
        <div class="bg-slate-800 p-4 rounded-2xl border border-slate-700">
            <h3 class="text-xs font-bold text-slate-400 uppercase tracking-wider mb-3">Historial Detallado de Revisiones</h3>
            <div id="listaRevisiones" class="space-y-3"></div>
        </div>

    </div>

    <script>
        let chartInstance = null;

        // Función para normalizar nombres de marcas y obtener la URL del logo
        function getLogoUrl(marca) {
            if (!marca) return '';
            let slug = marca.toLowerCase().trim()
                .replace(/[\s\-_]+/g, '-')
                .replace(/[^a-z0-9\-]/g, '');

            if (slug.includes('mercedes')) slug = 'mercedes-benz';
            if (slug.includes('great') && slug.includes('wall')) slug = 'great-wall';

            return `https://raw.githubusercontent.com/filippofilip95/car-logos-dataset/master/logos/thumb/${slug}.png`;
        }

        async function buscarPatente() {
            const patente = document.getElementById('patenteInput').value.trim();
            if (!patente) return alert("Ingresa una patente válida");

            const res = await fetch(`/api/buscar/${patente}`);
            const data = await res.json();

            if (data.error) {
                alert("No se encontraron registros para esta patente.");
                return;
            }

            document.getElementById('resultados').classList.remove('hidden');

            const info = data.info_vehiculo;
            document.getElementById('vehiculoTitulo').innerText = `${info.marca || ''} ${info.modelo || ''} ${info.ano ? '('+info.ano+')' : ''}`;
            document.getElementById('numMotor').innerText = info.num_motor || 'N/A';
            document.getElementById('numChasis').innerText = info.num_chasis || 'N/A';

            // Cargar Logo de Marca
            const logoContainer = document.getElementById('logoContainer');
            const imgLogo = document.getElementById('marcaLogo');
            if (info.marca) {
                logoContainer.style.display = 'flex';
                imgLogo.src = getLogoUrl(info.marca);
            } else {
                logoContainer.style.display = 'none';
            }

            document.getElementById('cantAprobadas').innerText = data.aprobadas;
            document.getElementById('cantRechazadas').innerText = data.rechazadas;
            document.getElementById('tasaAprobacion').innerText = `${data.tasa_aprobacion}%`;

            const alertaBox = document.getElementById('alertaFraude');
            if (data.alertas && data.alertas.length > 0) {
                alertaBox.classList.remove('hidden');
                document.getElementById('alertaTexto').innerText = data.alertas[0].mensaje;
            } else {
                alertaBox.classList.add('hidden');
            }

            const lista = document.getElementById('listaRevisiones');
            lista.innerHTML = '';
            data.historial.forEach(rev => {
                const item = document.createElement('div');
                item.className = 'bg-slate-900/80 p-3 rounded-xl border border-slate-700/50 text-xs';
                item.innerHTML = `
                    <div class="flex justify-between items-center mb-1">
                        <span class="px-2 py-0.5 rounded text-[10px] font-bold ${rev.resultado.includes('APROBADO') ? 'bg-emerald-500/20 text-emerald-400 border border-emerald-500/30' : 'bg-red-500/20 text-red-400 border border-red-500/30'}">${rev.resultado}</span>
                        <span class="text-slate-400">${rev.fecha} • Planta: ${rev.planta || 'PRT'}</span>
                    </div>
                    <div class="flex justify-between text-slate-300 font-semibold mt-1">
                        <span>Certificado: ${rev.certificado || 'N/A'}</span>
                        <span class="text-amber-400">${Number(rev.kilometraje).toLocaleString()} km</span>
                    </div>
                `;
                lista.appendChild(item);
            });

            renderGrafico(data.historial);
        }

        function renderGrafico(historial) {
            const ctx = document.getElementById('kmChart').getContext('2d');
            const labels = historial.map(h => h.fecha);
            const dataKm = historial.map(h => h.kilometraje);

            if (chartInstance) chartInstance.destroy();

            chartInstance = new Chart(ctx, {
                type: 'line',
                data: {
                    labels: labels,
                    datasets: [{
                        label: 'Kilometraje',
                        data: dataKm,
                        borderColor: '#f59e0b',
                        backgroundColor: 'rgba(245, 158, 11, 0.1)',
                        borderWidth: 3,
                        tension: 0.3,
                        fill: true,
                        pointBackgroundColor: '#f59e0b'
                    }]
                },
                options: {
                    responsive: true,
                    plugins: { legend: { display: false } },
                    scales: {
                        x: { ticks: { color: '#94a3b8', font: { size: 9 } } },
                        y: { ticks: { color: '#94a3b8', font: { size: 9 } } }
                    }
                }
            });
        }
    </script>
</body>
</html>
"""

@app.get("/", response_class=HTMLResponse)
def index():
    return HTML_CODE

@app.get("/api/buscar/{patente}")
def buscar_patente(patente: str):
    if not os.path.exists(DB_PATH):
        return {"error": "Base de datos no encontrada"}

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    ppu = patente.upper().replace('-', '').strip()

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tablas = [t[0] for t in cursor.fetchall()]
    tabla_principal = tablas[0] if tablas else "revisiones"

    cursor.execute(f"PRAGMA table_info({tabla_principal})")
    columnas = [c[1].lower() for c in cursor.fetchall()]
    col_ppu = next((c for c in columnas if 'ppu' in c or 'patente' in c), columnas[0])

    cursor.execute(f"SELECT * FROM {tabla_principal} WHERE {col_ppu} = ?", (ppu,))
    filas = cursor.fetchall()

    if not filas:
        return {"error": "Patente no encontrada"}

    historial = []
    alertas = []
    ultimo_km = None
    fecha_anterior = None
    aprobadas = 0
    rechazadas = 0
    col_map = {name: idx for idx, name in enumerate(columnas)}

    for f in filas:
        fec_rev = str(f[col_map.get('fec_revision', 0)])[:10]
        planta = f[col_map.get('cod_prt', 0)] if 'cod_prt' in col_map else ''
        cert = f[col_map.get('num_certificado', 0)] if 'num_certificado' in col_map else ''
        km = f[col_map.get('kilometraje', 0)] if 'kilometraje' in col_map else 0
        try: km = int(km)
        except: km = 0

        resultado = "APROBADO"
        aprobadas += 1

        if km > 0:
            if ultimo_km is not None and km < ultimo_km:
                diferencia = ultimo_km - km
                alertas.append({
                    "mensaje": f"Kilometraje disminuyó en {diferencia:,} km entre {fecha_anterior} ({ultimo_km:,} km) y {fec_rev} ({km:,} km)."
                })
            ultimo_km = km
            fecha_anterior = fec_rev

        historial.append({
            "fecha": fec_rev,
            "planta": planta,
            "certificado": cert,
            "kilometraje": km,
            "resultado": resultado
        })

    info_vehiculo = {
        "marca": f[col_map.get('marca', 0)] if 'marca' in col_map else '',
        "modelo": f[col_map.get('modelo', 0)] if 'modelo' in col_map else '',
        "ano": f[col_map.get('ano_fab', 0)] if 'ano_fab' in col_map else '',
        "num_motor": f[col_map.get('num_motor', 0)] if 'num_motor' in col_map else '',
        "num_chasis": f[col_map.get('num_chasis', 0)] if 'num_chasis' in col_map else ''
    }

    return {
        "ppu": ppu,
        "info_vehiculo": info_vehiculo,
        "aprobadas": aprobadas,
        "rechazadas": rechazadas,
        "tasa_aprobacion": 100,
        "alertas": alertas,
        "historial": historial
    }

def run_fastapi():
    uvicorn.run(app, host="127.0.0.1", port=8000)

thread = threading.Thread(target=run_fastapi, daemon=True)
thread.start()
time.sleep(3)

print("\n📌 Paso 4: Generando enlace público...")
proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

url_encontrada = None
for _ in range(40):
    line = proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url_encontrada = match.group(0)
        break
    time.sleep(0.5)

if url_encontrada:
    print("\n" + "="*60)
    print("🚀 ¡ENLACE CON LOGOS DE MARCAS LISTO!")
    print("="*60)
    print(f"\n👉  {url_encontrada}  👈\n")
    print("Abre el enlace en Chrome desde tu teléfono Android.")
    print("============================================================\n")

<>:146: SyntaxWarning: invalid escape sequence '\s'
<>:146: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_12036/1424999286.py:146: SyntaxWarning: invalid escape sequence '\s'
  .replace(/[\s\-_]+/g, '-')


📌 Paso 1: Conectando a Google Drive...
Mounted at /content/drive

📌 Paso 2: Verificando base de datos...
   Descomprimiendo en segundo plano...
   ¡Descompresión completada!

📌 Paso 3: Lanzando Servidor de la App con Logos...


INFO:     Started server process [12036]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)



📌 Paso 4: Generando enlace público...

🚀 ¡ENLACE CON LOGOS DE MARCAS LISTO!

👉  https://moore-cable-usually-adjust.trycloudflare.com  👈

Abre el enlace en Chrome desde tu teléfono Android.

